# Práctica — Clase 9: completa el `text_to_speech` del asistente de voz

En el proyecto real de IBM `chatapp-with-voice-and-openai-outline`, el
archivo `worker.py` tiene tres funciones: `speech_to_text`,
`openai_process_message` y `text_to_speech`. Las dos primeras están
completas en el material fuente — pero `text_to_speech` se quedó con un
`base_url = '...'` de relleno, sin terminar de conectar con el servicio de
síntesis de voz (Watson TTS, de pago).

En esta práctica vas a resolver tú ese mismo hueco, usando `gTTS`
(Google Text-to-Speech), la alternativa 100% gratuita que usamos en el
notebook de clase. El resto del pipeline (STT con Whisper, el LLM con
SmolLM2) ya está resuelto para que te concentres en la pieza que falta.

In [ ]:
# Instalamos las librerías necesarias.
!pip install -q gTTS transformers accelerate

## Ejercicio: completa `text_to_speech`

Abajo tienes el andamiaje completo del pipeline. Las funciones
`speech_to_text` y `llm_process_message` ya están resueltas (igual que en el
`worker.py` real). Tu tarea es completar `text_to_speech` siguiendo las
instrucciones de los comentarios `# TODO`.

Pista: `gTTS` se usa así — `gTTS(text=texto, lang=idioma)` crea el objeto de
audio, y `.save(ruta)` lo guarda como archivo `.mp3`.

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
from gtts import gTTS
from IPython.display import Audio

# --- Ya resuelto: Speech-to-Text ---
stt_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")

def speech_to_text(audio_path):
    resultado = stt_pipe(audio_path)
    return resultado["text"].strip()


# --- Ya resuelto: el LLM ---
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def llm_process_message(user_message, system_prompt="Eres un asistente breve y amable."):
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    entrada = tokenizer.apply_chat_template(
        mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    with torch.inference_mode():
        salida = llm_model.generate(entrada, max_new_tokens=80, do_sample=True, temperature=0.7)
    texto_generado = tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True)
    return texto_generado.strip()


# --- TODO: completa esta función ---
# Esta es la función que, en el worker.py real del material de IBM, se quedó
# con un placeholder "base_url = '...'" sin terminar.
def text_to_speech(text, salida_path="respuesta.mp3", idioma="es"):
    """Debe convertir `text` en un archivo de audio guardado en `salida_path`
    y devolver `salida_path`."""
    # TODO: crea un objeto gTTS con el texto y el idioma indicados
    voz = None  # <-- reemplaza esto

    # TODO: guarda el audio generado en salida_path usando voz.save(...)

    # TODO: devuelve salida_path
    pass

## Probando tu solución

Genera un audio de ejemplo, transcríbelo, pásalo por el LLM y usa tu
`text_to_speech` para escuchar la respuesta. Si `text_to_speech` está bien
completada, la celda de abajo debería reproducir un audio con la respuesta
del asistente.

In [ ]:
gTTS(text="What is the capital of France?", lang="en").save("audio_usuario.mp3")

texto_transcrito = speech_to_text("audio_usuario.mp3")
print("Texto transcrito:", texto_transcrito)

respuesta = llm_process_message(texto_transcrito)
print("Respuesta del LLM:", respuesta)

ruta_audio_respuesta = text_to_speech(respuesta, idioma="en")
Audio(ruta_audio_respuesta)

<details>
<summary>💡 Ver solución</summary>

```python
def text_to_speech(text, salida_path="respuesta.mp3", idioma="es"):
    """Convierte texto en un archivo de audio y devuelve la ruta del archivo."""
    # Creamos el objeto gTTS con el texto y el idioma pedidos.
    voz = gTTS(text=text, lang=idioma)

    # Guardamos el audio generado en la ruta indicada.
    voz.save(salida_path)

    # Devolvemos la ruta del archivo para que quien llame a la función
    # pueda reproducirlo o enviarlo de vuelta al frontend.
    return salida_path
```

Esto es, en esencia, lo mismo que hace la versión de pago con Watson TTS:
recibe un texto, genera un archivo de audio, y devuelve una referencia a ese
audio (aquí una ruta local; en `worker.py` real, los bytes del audio
devueltos por la API de Watson).
</details>

## Para seguir

- Notebook de clase: `notebooks_clase/clase09_asistente_voz_conversacional.ipynb`
- Capítulo del manual: `manual/13_asistente_voz_conversacional.md`

Reto extra (opcional): cambia el `system_prompt` de `llm_process_message`
para que el asistente responda como un terapeuta, un mecánico, un
cuentacuentos o un profesor — igual que sugieren los ejercicios de "prompt
engineering" del material original de IBM — y observa cómo cambia el tono de
la respuesta (y, por tanto, del audio generado).